# SMS Spam Inference and Error Analysis

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

In [2]:
spark = SparkSession.builder.appName('sms-spam-inference').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

model_path = '/content/artifacts/models/sms_spam_best_pipeline'
model = PipelineModel.load(model_path)
clean_df = spark.read.parquet('/content/artifacts/clean_sms.parquet')

print('Model path:', model_path)
print('Rows:', clean_df.count())
clean_df.show(5, truncate=False)

Model path: /content/artifacts/models/sms_spam_best_pipeline
Rows: 5572
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|label|label_text|message                                                                                                                                                    |msg_len_chars|msg_len_words|
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|0    |ham       |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                            |111          |20           |
|0    |ham       |Ok lar... Joking wif u oni...                                                                     

In [3]:
scored_df = model.transform(clean_df.select('label', 'message'))

if 'probability' in scored_df.columns:
    scored_df = scored_df.withColumn('score_array', vector_to_array('probability'))
    scored_df = scored_df.withColumn('spam_score', F.col('score_array')[1])
else:
    scored_df = scored_df.withColumn('score_array', vector_to_array('rawPrediction'))
    scored_df = scored_df.withColumn('margin', F.col('score_array')[1] - F.col('score_array')[0])
    scored_df = scored_df.withColumn('spam_score', 1 / (1 + F.exp(-F.col('margin'))))

scored_df.select('label', 'prediction', 'spam_score', 'message').show(10, truncate=False)

uncertain_df = scored_df.withColumn('uncertainty', F.abs(F.col('spam_score') - F.lit(0.5))).orderBy('uncertainty')
uncertain_df.select('label', 'prediction', 'spam_score', 'message').show(15, truncate=False)

+-----+----------+--------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|spam_score          |message                                                                                                                                                         |
+-----+----------+--------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0    |0.0       |0.053150053848377114|Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                                 |
|0    |0.0       |0.05099208133267921 |Ok lar... Joking wif u oni...                                                                                                                                

In [4]:
from pathlib import Path

false_pos = scored_df.filter((F.col('label') == 0) & (F.col('prediction') == 1)).select('spam_score', 'message')
false_neg = scored_df.filter((F.col('label') == 1) & (F.col('prediction') == 0)).select('spam_score', 'message')

print('False positives:', false_pos.count())
print('False negatives:', false_neg.count())

artifact_root = Path('/content/artifacts')
false_pos.toPandas().to_csv(artifact_root / 'false_positives.csv', index=False)
false_neg.toPandas().to_csv(artifact_root / 'false_negatives.csv', index=False)
print('Saved error files to:', artifact_root)

false_pos.orderBy(F.desc('spam_score')).show(10, truncate=False)
false_neg.orderBy(F.asc('spam_score')).show(10, truncate=False)

False positives: 17
False negatives: 21
Saved error files to: /content/artifacts
+------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------